# Data Agent Industrial 
## (LangGraph + Multi-Agent + RAG + Guardrails + Streamlit + LangSmith)

                ┌──────────────────────┐
                │   User (Streamlit)   │
                └─────────┬────────────┘
                          ↓
                ┌──────────────────────┐
                │   LangGraph Router   │
                └─────────┬────────────┘
                          ↓
     ┌─────────────────────────────────────────┐
     │             MULTI-AGENTS               │
     │                                         │
     │  🧠 SQL Agent (Text-to-SQL)            │
     │  📊 Analytics Agent (O&G KPIs)         │
     │  🧾 Memory Agent (RAG / context)       │
     │  🧪 Debug Agent (fix SQL errors)       │
     └──────────────┬─────────────────────────┘
                    ↓
           ┌───────────────────┐
           │ SQLite / DuckDB   │
           └─────────┬─────────┘
                     ↓
        ┌────────────────────────────┐
        │ Final Answer Generator LLM │
        └────────────────────────────┘
                     ↓
        ┌────────────────────────────┐
        │ LangSmith Observability   │
        └────────────────────────────┘

---

### LLM (OLLAMA - LLAMA 3.1)

In [75]:
# from langchain_community.chat_models import ChatOllama
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)

### BANCO (SQLite / Produção de Petróleo)

---
## MULTI-AGENTS (CORE SYSTEM)
---

### SQL AGENT (TEXT → SQL REAL)

In [76]:
import sqlite3

conn = sqlite3.connect("oil.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS well_production (
    well_name TEXT,
    field_name TEXT,
    production_date TEXT,
    oil_bbl FLOAT,
    gas_mscf FLOAT,
    water_bbl FLOAT,
    hours_on FLOAT
)
""")

In [77]:
data = [
('WELL-A1','FIELD-X','2026-06-01',1200,800,300,24),
('WELL-A2','FIELD-X','2026-06-01',900,600,500,24),
('WELL-B1','FIELD-Y','2026-06-01',1500,1100,200,24),
('WELL-B2','FIELD-Y','2026-06-01',700,400,800,24),
('WELL-A1','FIELD-X','2026-06-02',1100,750,350,24),
('WELL-A2','FIELD-X','2026-06-02',950,650,480,24),
('WELL-B1','FIELD-Y','2026-06-02',1400,1000,250,24),
('WELL-B2','FIELD-Y','2026-06-02',650,380,900,24),
]

cursor.executemany("""
INSERT INTO well_production VALUES (?,?,?,?,?,?,?)
""", data)

conn.commit()

In [78]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
# from langchain.agents import create_sql_agent

# O caminho correto do import mudou para o submódulo abaixo:
from langchain_community.agent_toolkits.sql.base import create_sql_agent

db = SQLDatabase.from_uri("sqlite:///oil.db")

sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)

sql_agent = create_sql_agent(
    llm=llm,
    toolkit=sql_toolkit,
    verbose=True
)

### ANALYTICS AGENT (OIL & GAS KPIs)

In [79]:
class AnalyticsAgent:

    def water_cut_sql(self):
        return """
        SELECT well_name,
               SUM(water_bbl)*1.0 / SUM(oil_bbl + water_bbl) AS water_cut
        FROM well_production
        GROUP BY well_name;
        """

    def decline_sql(self):
        return """
        SELECT well_name,
               MAX(oil_bbl) - MIN(oil_bbl) AS decline
        FROM well_production
        GROUP BY well_name;
        """

### MEMORY AGENT (RAG SIMPLES)

In [80]:
from collections import deque

class MemoryAgent:
    def __init__(self):
        self.memory = deque(maxlen=20)

    def add(self, text):
        self.memory.append(text)

    def get(self):
        return "\n".join(self.memory)

### DEBUG AGENT (corrige SQL quebrado)

In [81]:
class DebugAgent:

    def fix_sql(self, error, query):
        prompt = f"""
        Corrija o SQL abaixo:

        SQL: {query}
        ERRO: {error}

        Retorne apenas SQL válido.
        """

        return llm.invoke(prompt).content

---
## LANGGRAPH (ORQUESTRADOR MULTI-AGENT)
---

### STATE

In [82]:
from typing import TypedDict, Optional

class AgentState(TypedDict):
    question: str
    sql: Optional[str]
    result: Optional[str]
    final: Optional[str]
    error: Optional[str]

### SQL NODE

In [83]:
def sql_node(state: AgentState):
    try:
        result = sql_agent.run(state["question"])
        return {"result": result}

    except Exception as e:
        return {"error": str(e)}

### ANALYTICS NODE

In [84]:
def analytics_node(state: AgentState):

    agent = AnalyticsAgent()
    question = state["question"].lower()

    if "water cut" in question:
        sql = agent.water_cut_sql()
    else:
        sql = agent.decline_sql()

    import pandas as pd
    df = pd.read_sql_query(sql, conn)

    return {"result": df.to_string()}

### MEMORY NODE

In [85]:
memory = MemoryAgent()

def memory_node(state: AgentState):
    memory.add(state["question"])
    return {"memory": memory.get()}

### FINAL NODE

In [86]:
def final_node(state: AgentState):

    prompt = f"""
    Você é um engenheiro de produção de petróleo.

    Contexto:
    {state.get("memory","")}

    Resultado:
    {state["result"]}

    Gere uma resposta clara e técnica.
    """

    response = llm.invoke(prompt).content

    return {"final": response}

---
## LANGGRAPH FLOW
---

In [87]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

### nodes

In [88]:
graph.add_node("sql", sql_node)
graph.add_node("analytics", analytics_node)
graph.add_node("memory", memory_node)
graph.add_node("final", final_node)

### entry

In [89]:
graph.set_entry_point("memory")

### routing simples

In [90]:
def router(state: AgentState):
    q = state["question"].lower()

    if "water cut" in q or "decline" in q:
        return "analytics"

    return "sql"

In [91]:
graph.add_conditional_edges(
    "memory",
    router,
    {
        "sql": "sql",
        "analytics": "analytics"
    }
)

graph.add_edge("sql", "final")
graph.add_edge("analytics", "final")
graph.add_edge("final", END)

### compile

In [92]:
app = graph.compile()

### EXECUÇÃO

In [98]:
result = app.invoke({
    "question": "Qual poço teve maior produção de óleo?"
})

print(result["final"])



> Entering new SQL Agent Executor chain...


KeyboardInterrupt: 

In [94]:
cursor2 = conn.cursor()

cursor2.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

print(cursor2.fetchall())

[('well_production',)]


In [95]:
import pandas as pd

df = pd.read_sql_query(
    "SELECT * FROM well_production",
    conn
)

print(df)

   well_name field_name production_date  oil_bbl  gas_mscf  water_bbl  \
0    WELL-A1    FIELD-X      2026-06-01   1200.0     800.0      300.0   
1    WELL-A2    FIELD-X      2026-06-01    900.0     600.0      500.0   
2    WELL-B1    FIELD-Y      2026-06-01   1500.0    1100.0      200.0   
3    WELL-B2    FIELD-Y      2026-06-01    700.0     400.0      800.0   
4    WELL-A1    FIELD-X      2026-06-02   1100.0     750.0      350.0   
5    WELL-A2    FIELD-X      2026-06-02    950.0     650.0      480.0   
6    WELL-B1    FIELD-Y      2026-06-02   1400.0    1000.0      250.0   
7    WELL-B2    FIELD-Y      2026-06-02    650.0     380.0      900.0   
8    WELL-A1    FIELD-X      2026-06-01   1200.0     800.0      300.0   
9    WELL-A2    FIELD-X      2026-06-01    900.0     600.0      500.0   
10   WELL-B1    FIELD-Y      2026-06-01   1500.0    1100.0      200.0   
11   WELL-B2    FIELD-Y      2026-06-01    700.0     400.0      800.0   
12   WELL-A1    FIELD-X      2026-06-02   1100.0   

In [96]:
response = llm.invoke("""
Você é especialista em SQL e produção de petróleo.

Tabela:

well_production(
    well_name,
    field_name,
    production_date,
    oil_bbl,
    gas_mscf,
    water_bbl,
    hours_on
)

Regras:

- Quando a pergunta mencionar "maior produção",
  considere a produção acumulada por poço.
- Use GROUP BY quando necessário.
- Retorne apenas SQL.
- Nunca explique o SQL.

Pergunta:

Qual poço teve maior produção de óleo?
""")

print(response.content)

SELECT well_name FROM well_production GROUP BY well_name ORDER BY SUM(oil_bbl) DESC LIMIT 1


In [97]:
import pandas as pd

sql = """
SELECT well_name
FROM well_production
GROUP BY well_name
ORDER BY SUM(oil_bbl) DESC
LIMIT 1
"""

df = pd.read_sql_query(sql, conn)

print(df)

  well_name
0   WELL-B1


In [99]:
print(type(sql_agent))

<class 'langchain_classic.agents.agent.AgentExecutor'>


In [100]:
print(sql_agent.tools)

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7d5ee7bf6050>), InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7d5ee7bf6050>), ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7d5ee7bf6050>), QuerySQLCheckerTool(description='Use this tool to double check if

In [101]:
result = sql_agent.invoke({
    "input": "Qual poço teve maior produção de óleo?"
})

print(result)



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: well_productionI'll list the tables in the database and then check the schema of the relevant table.

Action: sql_db_list_tables
Action Input: well_productionI will follow the instructions to interact with the SQL database.

Question: Qual poço teve maior produção de óleo?
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: well_productionI'll follow the instructions.

Question: Qual poço teve maior produção de óleo?
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: well_productionI'll follow the instructions.

Question: Qual poço teve maior produção de óleo?
Thought: I need to find out which table(s) contain information about oil production. Let m

In [102]:
print(llm.invoke("""
You are an agent.

Question:
What is 2 + 2?

Respond EXACTLY in this format:

Thought: your reasoning
Final Answer: your answer
""").content)

Thought: I recall the basic arithmetic operations and know that addition involves combining two or more numbers. In this case, we have two instances of the number 2.

Final Answer: 4


In [103]:
print(sql_agent.max_iterations)

15


In [104]:
print(sql_agent.agent)

runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': "sql_db_query - Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.\nsql_db_schema - Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3\nsql_db_list_tables - Input is an empty string, output is a comma-separated list of tables in the database.\nsql_db_query_checker - Use this tool to dou

In [105]:
for tool in sql_agent.tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [106]:
for tool in sql_agent.tools:
    if tool.name == "sql_db_list_tables":
        print(tool.invoke(""))

well_production


In [107]:
for tool in sql_agent.tools:
    if tool.name == "sql_db_schema":
        print(tool.invoke("well_production"))


CREATE TABLE well_production (
	well_name TEXT, 
	field_name TEXT, 
	production_date TEXT, 
	oil_bbl FLOAT, 
	gas_mscf FLOAT, 
	water_bbl FLOAT, 
	hours_on FLOAT
)

/*
3 rows from well_production table:
well_name	field_name	production_date	oil_bbl	gas_mscf	water_bbl	hours_on
WELL-A1	FIELD-X	2026-06-01	1200.0	800.0	300.0	24.0
WELL-A2	FIELD-X	2026-06-01	900.0	600.0	500.0	24.0
WELL-B1	FIELD-Y	2026-06-01	1500.0	1100.0	200.0	24.0
*/


In [108]:
prompt = """
You are an agent.

Available tools:

sql_db_list_tables
sql_db_schema
sql_db_query
sql_db_query_checker

Question:
Qual poço teve maior produção de óleo?

You already know that the database contains:

well_production

What should be your next action?

Respond only with:

Action: <tool_name>
Action Input: <input>
"""

response = llm.invoke(prompt)

print(response.content)

Action: sql_db_query
Action Input: SELECT * FROM well_production ORDER BY production DESC LIMIT 1


In [109]:
response = llm.invoke("""
Question: Qual poço teve maior produção de óleo?

Thought: I should look at the tables in the database.

Action: sql_db_list_tables
Action Input:

Observation: well_production

Continue from here using exactly the ReAct format.
""")

print(response.content)

**Step 1: Identify the Goal**
The goal is to find which well had the highest production of oil.

**Step 2: Determine the Relevant Data**
To solve this, we need data on oil production for each well. This likely involves querying a database that contains information about oil wells and their production rates.

**Step 3: Choose the Right Tool**
Given the mention of looking at tables in a database, it seems like SQL (Structured Query Language) is the appropriate tool for this task. We will use SQL commands to query the database.

**Step 4: Write the SQL Command**
To find which well had the highest production of oil, we would write an SQL command that selects the well with the maximum oil production from the relevant table in the database. The exact command might look something like:
```sql
SELECT well_name, MAX(oil_production) 
FROM well_production 
GROUP BY well_name;
```
However, this is a simplified example and actual implementation may vary based on the structure of the database.

**St

In [111]:
import langchain
import langchain_community

print("langchain:", langchain.__version__)
print("langchain_community:", langchain_community.__version__)

langchain: 1.3.4
langchain_community: 0.4.2


In [112]:
from langgraph.prebuilt import create_react_agent

print(create_react_agent)

<function create_react_agent at 0x7d5f24081e40>


In [113]:
from langchain.agents import create_agent

print(create_agent)

<function create_agent at 0x7d5f26a7f9c0>


In [114]:
import inspect
from langchain.agents import create_agent

print(inspect.signature(create_agent))

(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None, transformers: 'Sequence[TransformerFactory] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'


In [115]:
print(type(sql_toolkit.get_tools()))
print(len(sql_toolkit.get_tools()))

<class 'list'>
4
